<a href="https://colab.research.google.com/github/INNORH/FlyRank-Internship/blob/main/notebooks/02_your_first_readable_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/INNORH/FlyRank-Internship/blob/main/notebooks/02_your_first_readable_model.ipynb)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [ ]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [ ]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Look closely: the tree **wins at Precision@50** but your hand rule **wins at Precision@20**. Both results are real. A sharp human rule can be excellent at the very top of the list; the model's advantage shows up deeper, where simple rules run out of signal. Saying exactly that — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [ ]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [ ]:
# Your experiment here
I ran the full sequence: hand rule → depth-2 tree → deeper trees → feature swap → the leakage demo → then re-checked everything with a proper client-holdout split, since the notebook flags that the in-sample numbers are for teaching only.

1. Hand rule baseline (in-sample)
My rule — stale AND visible, scored by impression volume — got:

Precision@20: 0.950
Precision@50: 0.660

That's a strong rule at the very top. It's basically hand-picking the highest-traffic pages that also haven't been touched in 6+ months, and in-sample that combination is almost always right.

2. Depth-2 tree (in-sample)
The tree ended up reading as: if impressions_90d > 5.5 and content_age_days ≤ 312.5 → declining. That's it — three splits, and it basically rediscovered "moderately-trafficked, not-too-old pages are the ones sliding."

Precision@20: 0.650
Precision@50: 0.640

So in-sample, my hand rule actually beat the tree at the top of the list (0.95 vs 0.65) but lost slightly deeper (0.66 vs 0.64) — mirrors what the notebook predicted: sharp human rules win at the very top, models find signal deeper where hand-tuned thresholds run out.

3. Depth 3 and 4 — does more depth help, and can I still read it?

Depth-3: Precision@20 = 0.650, Precision@50 = 0.660, 8 leaves. Still readable — it just adds a ctr split and an avg_position split under the existing branches.
Depth-4: Precision@20 = 0.700, Precision@50 = 0.720, 16 leaves. This is where it starts to strain — I had to trace through nested splits like content_age_days ≤ 108.5 → days_since_last_update ≤ 14 → class 1 to explain a single leaf. Technically readable, but no longer something I'd screenshot into a report without annotating it.

So yes, precision improved with depth (0.64 → 0.66 → 0.72 at P@50), but the "read it out loud" property degrades fast after depth 2-3. That trade-off felt like the actual lesson here — more depth isn't free even when it's not overfitting in the classic sense.

4. Feature swap — drop impressions_90d, add engagement_rate
With impressions_90d removed and engagement_rate added, the tree's first split changed completely: it now splits on avg_position first, then content_age_days. Feature importances confirm this — avg_position (0.60) and content_age_days (0.40) did all the work; engagement_rate got zero importance, meaning the tree never found it useful even when offered. Precision@20 actually went up slightly (0.70 vs 0.65) but Precision@50 dipped a bit (0.66 vs 0.64→ roughly flat). My takeaway: impressions_90d wasn't uniquely necessary — avg_position picked up the slack — but engagement_rate on its own isn't pulling weight in this label.

5. The leakage demo
Feeding in trend_pct (the value the down label is literally bucketed from) gave a tree that split purely on trend_pct ≤ -20.05 and hit Precision@20 = 1.000, Precision@50 = 1.000. Perfect score, zero information gained — it's the answer key with extra steps. This was the clearest "aha" of the week: a suspiciously perfect number is a leakage alarm, not a win.

6. The real test — client-holdout split
This is the part that mattered most. I split by client_id (24 clients train / 8 test, zero overlap) and re-scored everything on the held-out clients only:

	P@20	P@50
Hand rule	0.500	0.560
Depth-2 tree	0.600	0.560
Depth-3 tree	0.550	0.620
Depth-4 tree	0.500	0.580

The gap did not hold. In-sample, my hand rule looked dominant at the top (0.95 P@20). On genuinely unseen clients, it drops to 0.50 — barely better than a coin flip — while the depth-2 tree actually edges it out (0.60). The in-sample "hand rule wins at the top" story was partly an illusion of having already seen those exact pages. Depth-3 was the best performer at P@50 out-of-sample (0.62), and depth-4 — despite winning in-sample — didn't hold its advantage once tested on new clients, which is a decent small example of overfitting on rule-of-thumb complexity even at "just" 16 leaves.

Takeaway (observed, not proven): hand-written rules can look deceptively strong when evaluated on the same data they were tuned against. A shallow, readable tree evaluated honestly on held-out clients is a fairer comparison — and here it modestly beat the hand rule, though not by the dramatic margin the in-sample numbers implied. The leakage test is the one unambiguous result: a feature derived from the label will always look perfect and always teaches you nothing.

No client-identifying info used or included above.

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.